# Lab 1: Exploring Two Related Datasets
Suyang Pan

In [14]:
import duckdb
con = duckdb.connect()
print('DuckDB is ready!')

DuckDB is ready!


In [15]:
from pathlib import Path

# Locate the project directory from the repository root or notebooks folder.
root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

urban_path = root / "data" / "urban_population.csv"
countries_path = root / "data" / "countries.csv"

# Verify that both input files exist.
assert urban_path.is_file(), f"File not found: {urban_path}"
assert countries_path.is_file(), f"File not found: {countries_path}"

# Load both CSV files and recognize common missing-value markers.
con.sql(f"""
CREATE OR REPLACE TABLE urban AS
SELECT *
FROM read_csv(
    '{urban_path.as_posix()}',
    nullstr = ['', 'NA', 'N/A']
)
""")

con.sql(f"""
CREATE OR REPLACE TABLE countries AS
SELECT *
FROM read_csv(
    '{countries_path.as_posix()}',
    nullstr = ['', 'NA', 'N/A']
)
""")

# Report the number of rows in each table.
display(con.sql("""
SELECT 'urban' AS table_name, COUNT(*) AS row_count
FROM urban
UNION ALL
SELECT 'countries' AS table_name, COUNT(*) AS row_count
FROM countries
""").df())

# Inspect column names and inferred data types.
display(con.sql("DESCRIBE urban").df())
display(con.sql("DESCRIBE countries").df())

,table_name,row_count
0,urban,6650
1,countries,265


,column_name,column_type,null,key,default,extra
0,country_name,VARCHAR,YES,None,None,None
1,country_code,VARCHAR,YES,None,None,None
2,year,BIGINT,YES,None,None,None
3,urban_pct,DOUBLE,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,Country Code,VARCHAR,YES,None,None,None
1,Region,VARCHAR,YES,None,None,None
2,IncomeGroup,VARCHAR,YES,None,None,None
3,SpecialNotes,VARCHAR,YES,None,None,None
4,TableName,VARCHAR,YES,None,None,None


In [16]:
# Profile column values, data types, and missingness.
display(con.sql("SUMMARIZE urban").df())
display(con.sql("SUMMARIZE countries").df())

# Compare distinct non-null country codes across the two tables.
display(con.sql("""
WITH urban_keys AS (
    SELECT DISTINCT country_code AS code
    FROM urban
    WHERE country_code IS NOT NULL
),
country_keys AS (
    SELECT DISTINCT "Country Code" AS code
    FROM countries
    WHERE "Country Code" IS NOT NULL
)
SELECT
    (SELECT COUNT(*) FROM urban_keys) AS urban_keys,
    (SELECT COUNT(*) FROM country_keys) AS country_keys,
    (
        SELECT COUNT(*)
        FROM urban_keys
        INNER JOIN country_keys USING (code)
    ) AS matched_keys,
    (
        SELECT COUNT(*)
        FROM urban_keys a
        WHERE NOT EXISTS (
            SELECT 1 FROM country_keys b WHERE b.code = a.code
        )
    ) AS urban_only,
    (
        SELECT COUNT(*)
        FROM country_keys b
        WHERE NOT EXISTS (
            SELECT 1 FROM urban_keys a WHERE a.code = b.code
        )
    ) AS countries_only
""").df())

# Identify records in the main table without matching metadata.
display(con.sql("""
SELECT DISTINCT u.country_code, u.country_name
FROM urban u
WHERE NOT EXISTS (
    SELECT 1
    FROM countries c
    WHERE c."Country Code" = u.country_code
)
ORDER BY u.country_code
""").df())

# Identify metadata records without matching main-table records.
display(con.sql("""
SELECT c."Country Code", c.TableName
FROM countries c
WHERE NOT EXISTS (
    SELECT 1
    FROM urban u
    WHERE u.country_code = c."Country Code"
)
ORDER BY c."Country Code"
""").df())

# Check whether country codes are unique in the metadata table.
display(con.sql("""
SELECT "Country Code", COUNT(*) AS row_count
FROM countries
GROUP BY "Country Code"
HAVING COUNT(*) > 1
ORDER BY row_count DESC
""").df())

# Count missing keys in both tables.
display(con.sql("""
SELECT
    'urban' AS table_name,
    COUNT(*) - COUNT(country_code) AS missing_keys
FROM urban
UNION ALL
SELECT
    'countries' AS table_name,
    COUNT(*) - COUNT("Country Code") AS missing_keys
FROM countries
""").df())


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,country_name,VARCHAR,Afghanistan,Zimbabwe,269,NaN,NaN,NaN,NaN,NaN,6650,0.00
1,country_code,VARCHAR,ABW,ZWE,269,NaN,NaN,NaN,NaN,NaN,6650,0.00
2,year,BIGINT,2000,2024,25,2012.0,7.211644800247497,2006,2012,2018,6650,0.00
3,urban_pct,DOUBLE,8.04381449833197,100.0,6247,58.11511404074759,22.988376471669298,39.19365627573303,57.77420704148115,76.87734970764477,6650,0.38


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Country Code,VARCHAR,ABW,ZWE,269,None,None,None,None,None,265,0.00
1,Region,VARCHAR,East Asia & Pacific,Sub-Saharan Africa,7,None,None,None,None,None,265,18.11
2,IncomeGroup,VARCHAR,High income,Upper middle income,4,None,None,None,None,None,265,18.11
3,SpecialNotes,VARCHAR,"22 countries, stretching from the westernmost ...",The reporting period for national accounts dat...,112,None,None,None,None,None,265,49.06
4,TableName,VARCHAR,Afghanistan,Zimbabwe,267,None,None,None,None,None,265,0.00


,urban_keys,country_keys,matched_keys,urban_only,countries_only
0,266,265,265,1,0


,country_code,country_name
0,INX,Not classified


,Country Code,TableName


,Country Code,row_count


,table_name,missing_keys
0,urban,0
1,countries,0


In [17]:
# Query 1: Size and shape
display(con.sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT country_code) AS distinct_codes,
    MIN(year) AS first_year,
    MAX(year) AS last_year
FROM urban
""").df())
print(
    "Interpretation: This reports the table's size and time coverage. "
    "The codes include economies and aggregate groups."
)

# Query 2: Filter
display(con.sql("""
SELECT country_name, year, ROUND(urban_pct, 2) AS urban_pct
FROM urban
WHERE country_code = 'CHN'
  AND year IN (2000, 2010, 2020, 2024)
ORDER BY year
""").df())
print(
    "Interpretation: This compares China's urban population share "
    "at four selected years."
)

# Query 3: Group
display(con.sql("""
SELECT
    Region,
    COUNT(*) AS metadata_records,
    COUNT(IncomeGroup) AS records_with_income_group
FROM countries
WHERE Region IS NOT NULL
GROUP BY Region
ORDER BY metadata_records DESC, Region
""").df())
print(
    "Interpretation: This counts metadata records by region and shows "
    "whether income classifications are available."
)

# Query 4: Rank
display(con.sql("""
SELECT
    country_name,
    ROUND(urban_pct, 2) AS urban_pct
FROM urban
WHERE year = 2024
  AND urban_pct IS NOT NULL
  AND country_code IN (
      SELECT "Country Code"
      FROM countries
      WHERE Region IS NOT NULL
  )
ORDER BY urban_pct DESC, country_name
LIMIT 10
""").df())
print(
    "Interpretation: This lists the ten economies with the highest "
    "reported urban population shares in 2024, excluding aggregate groups."
)

# Query 5: Quality check
display(con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(urban_pct) AS missing_values,
    ROUND(
        100.0 * (COUNT(*) - COUNT(urban_pct)) / COUNT(*), 2
    ) AS missing_pct,
    COUNT(*) - COUNT(DISTINCT (country_code, year))
        AS duplicate_key_rows,
    COUNT(*) FILTER (
        WHERE urban_pct < 0 OR urban_pct > 100
    ) AS out_of_range_values
FROM urban
""").df())
print(
    "Interpretation: This checks missing percentages, repeated "
    "country-year keys, and values outside the valid 0–100 range. "
    "Missing values are not treated as zero."
)

,row_count,distinct_codes,first_year,last_year
0,6650,266,2000,2024


Interpretation: This reports the table's size and time coverage. The codes include economies and aggregate groups.


,country_name,year,urban_pct
0,China,2000,36.43
1,China,2010,49.23
2,China,2020,63.52
3,China,2024,65.89


Interpretation: This compares China's urban population share at four selected years.


,Region,metadata_records,records_with_income_group
0,Europe & Central Asia,58,58
1,Sub-Saharan Africa,48,48
2,Latin America & Caribbean,42,42
3,East Asia & Pacific,37,37
4,Middle East & North Africa,23,23
5,South Asia,6,6
6,North America,3,3


Interpretation: This counts metadata records by region and shows whether income classifications are available.


,country_name,urban_pct
0,Bahrain,100.0
1,Bermuda,100.0
2,Cayman Islands,100.0
3,Gibraltar,100.0
4,"Hong Kong SAR, China",100.0
5,Kuwait,100.0
6,"Macao SAR, China",100.0
7,Monaco,100.0
8,Naoero,100.0
9,Singapore,100.0


Interpretation: This lists the ten economies with the highest reported urban population shares in 2024, excluding aggregate groups.


,total_rows,missing_values,missing_pct,duplicate_key_rows,out_of_range_values
0,6650,25,0.38,0,0


Interpretation: This checks missing percentages, repeated country-year keys, and values outside the valid 0–100 range. Missing values are not treated as zero.


## Data Preparation and Type Checks

The urban population indicator was downloaded from the World Bank and
reshaped from a wide table into a long table covering 2000–2024.
The main table contains 6,650 rows and four columns. The metadata table
contains 265 rows and five columns.

I checked both tables using DESCRIBE and SUMMARIZE. In the main table,
country names and codes are VARCHAR, year is BIGINT, and urban_pct is
DOUBLE. These types are appropriate. All five metadata columns are
text fields and were loaded as VARCHAR.

Empty fields and the markers NA and N/A were configured as NULL during
CSV loading. No additional type conversion was necessary. Missing
urban population shares were preserved rather than replaced with zero.

## Key Relationship

The shared key is urban.country_code = countries."Country Code".

The main table contains 266 distinct codes, while the metadata table
contains 265. There are 265 matching codes, one code found only in the
main table, and no codes found only in the metadata table. Neither table
has missing country codes, and the metadata table has no duplicate codes.

The relationship is many-to-one: each matching metadata record corresponds
to 25 annual records in the main table. Within the main table, the
combination of country_code and year uniquely identifies a row.

The unmatched code is INX, labeled "Not classified". Its label suggests
a classification-related aggregate rather than an individual economy.
The reason for its absence from the metadata requires confirmation from
the source. A left join would preserve these records with missing metadata;
an inner join would exclude them.

## Three Observations Before Trusting the Data

1. **Missing values require investigation.** There are 25 missing
   urban population shares out of 6,650 records, or 0.38%.
   Before comparing trends, I would identify which codes and years
   account for these missing values. Replacing them with zero would
   incorrectly imply that no residents live in urban areas.

2. **The key match is incomplete.** Only 265 of the 266 distinct
   main-table codes match the metadata. The unmatched INX code shows
   why join coverage must be checked explicitly. I would document its
   treatment before combining the tables.

3. **The ranking needs contextual interpretation.** All ten displayed
   economies in the 2024 ranking have an urban population share of
   100.0% after rounding to two decimal places. A high urban share does
   not by itself demonstrate better housing, infrastructure, or quality
   of life. I would check definitions and geographic coverage before
   drawing planning conclusions. The quality check found no duplicate
   country-year keys and no non-missing values outside 0–100, but these
   checks alone do not establish substantive accuracy.

## Question for a Non-Technical User

How has the share of people living in urban areas changed in China
between 2000 and 2024?

## Dataset Card 1: Urban Population Share

- **File:** `data/urban_population.csv`
- **Indicator:** Urban population (% of total population),
  `SP.URB.TOTL.IN.ZS`.
- **Source:** https://data.worldbank.org/indicator/SP.URB.TOTL.IN.ZS
- **Download:** https://api.worldbank.org/v2/en/indicator/SP.URB.TOTL.IN.ZS?downloadformat=csv
- **Producer and purpose:** The United Nations Population Division
  compiles urbanization estimates through World Urbanization Prospects.
  The World Bank distributes this indicator through World Development
  Indicators to support development monitoring and comparison.
- **License:** Creative Commons Attribution 4.0 (CC BY 4.0), as stated
  on the indicator page. This project attributes the source and
  reshapes the downloaded data.
- **Dimensions:** 6,650 rows × 4 columns.
- **Columns:** country_name, country_code, year, urban_pct.
- **Unit of observation:** One economy or aggregate code in one year.
- **Time coverage:** 2000–2024, selected from the downloaded series.
- **Frequency and updates:** Annual observations. Published values
  may be revised when the source estimates or WDI are updated.
  Annual observations do not imply that every value is newly measured
  each year.
- **Preparation:** Selected the 2000–2024 columns, reshaped the data
  from wide to long format, and renamed columns.
- **Limitations:** Urban definitions follow national statistical
  offices and may differ across countries. The table includes
  aggregate groups as well as individual economies. There are
  25 missing indicator values.

## Dataset Card 2: Country and Economy Metadata

- **File:** `data/countries.csv`
- **Name:** Country metadata accompanying the World Bank urban
  population indicator download.
- **Source:** The `Metadata_Country` CSV in the same download archive
  linked above.
- **Classification documentation:** https://datahelpdesk.worldbank.org/knowledgebase/articles/378834-how-does-the-world-bank-classify-countries
- **Producer and purpose:** The World Bank maintains economy codes,
  names, regions, income groups, and explanatory notes to support
  grouping and interpretation of development statistics.
- **License and terms:** Distributed with the World Bank WDI download.
  WDI is available under CC BY 4.0, subject to applicable dataset
  exceptions and the World Bank dataset terms:
  https://www.worldbank.org/en/about/legal/terms-of-use-for-datasets
- **Dimensions:** 265 rows × 5 columns.
- **Columns:** Country Code, Region, IncomeGroup, SpecialNotes, TableName.
- **Unit of observation:** One economy or aggregate metadata record.
- **Key:** Country Code; no duplicate or missing codes were found.
- **Time coverage:** A metadata snapshot accompanying the download,
  rather than a historical annual classification table.
- **Updates:** Income classifications are updated annually on July 1.
  Other metadata may change with source updates.
- **Preparation:** Removed the empty export column and preserved
  the five named metadata columns.
- **Limitations:** Current income classifications should not be
  interpreted as historical classifications for every year from
  2000 to 2024. Missing regional classifications require examination
  before using this table to distinguish economies from aggregates.
  INX ("Not classified") appears in the main table but has no matching
  metadata record.